Question Aswering chatbot using RAG

Install packages

In [ ]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 25.6 MB/s eta 0:00:00


In [ ]:
!pip install -q sentencepiece

UPLOAD pdf FILE

In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("Uploaded:", pdf_file)

Saving ssrn-5980914.pdf to ssrn-5980914.pdf
Uploaded: ssrn-5980914.pdf


Extract text from pdf

In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_file)

text = ""

for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
        text += page_text + "\n"

print("Characters:", len(text))

Characters: 236480


Text Chunking

In [ ]:
def create_chunks(text, chunk_size=500, overlap=100):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks

chunks = create_chunks(text)

print("Total Chunks:", len(chunks))

Total Chunks: 592


Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

print(embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

(592, 384)


Create FAISS DB

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings).astype("float32")
)

print("Stored vectors:", index.ntotal)

Stored vectors: 592


Retrieval Function

In [ ]:
def retrieve_context(query, top_k=3):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        top_k
    )

    retrieved = []

    for idx in indices[0]:
        retrieved.append(chunks[idx])

    return "\n".join(retrieved)

Load microsoft Phi2

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoConfig
)

model_name = "microsoft/phi-2"

config = AutoConfig.from_pretrained(model_name)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("Phi-2 Loaded Successfully")

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.7k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Phi-2 Loaded Successfully


RAG Generation Function

In [ ]:
def generate_answer(query):

    context = retrieve_context(query)

    prompt = f"""
You are a helpful assistant.

Use ONLY the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

Chat Interface

In [ ]:
while True:

    query = input("\nAsk Question: ")

    if query.lower() == "exit":
        break

    answer = generate_answer(query)

    print("\nAnswer:")
    print(answer)


Ask Question: agentic  ai definition


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer CodeGenTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Answer:

You are a helpful assistant.

Use ONLY the provided context.

Context:
elligence. 
What is Agentic AI? 
Agentic AI refers to artificial intelligence systems that are 
capable of autonomous goal -directed behaviour. Unlike 
conventional AI models that passively respond to inputs 
based on learned patterns, agentic AI systems actively initiate 
actions, plan trajectories toward objectives, monitor progress, 
and adapt strategies based on feedback from the environment. 
 
Figure 1.1: Components of agentic AI (source: Arion Research). 
 

CONCEPTS, DESIGN PATTERNS, AND
ilities define an agentic system? 

THE AGENTIC AI HANDBOOK 
 
Kaushik Bar 
 
• Why does this matter for the future of science, business, 
and society? 
 
These questions form the basis of this chapter. We begin 
by defining agentic AI in conceptual terms, then trace its 
historical emergence, and finally explore the key features that 
distinguish it from more conventional forms of artificial 
intelligence. 
What i